In [ ]:
import torch
from datasets import Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
import os
# 1. Expanded Dataset
raw_data = {
    "text": [
        # --- CLICK Actions (Label 0) ---
        "I click the login button", "Click on 'Submit'", "Tap the sign-in icon",
        "Press the cancel button", "I click on the 'Add to cart' link",
        "Click the search loupe", "Select the checkbox", "Click on the profile image",

        # --- TYPE Actions (Label 1) ---
        "I type 'Nour' in the name field", "Enter my email address",
        "Fill the search bar with 'Python'", "I input the password",
        "Type '123456' into the pin code field", "Input the text 'Test Automation'",

        # --- NAVIGATE Actions (Label 2) ---
        "I navigate to 'https://google.com'", "Go to the login page",
        "Open the url 'https://github.com'", "Navigate to the dashboard",
        "Browse to the settings section", "Visit 'www.facebook.com'",

        # --- VERIFY Actions (Label 3) ---
        "I should see 'Success' message", "Verify that the logo is visible",
        "Then I see the 'Error' notification", "The page title should be 'Home'",
        "I verify the existence of the 'Logout' button"
    ],
    "label": [0,0,0,0,0,0,0,0,  1,1,1,1,1,1,  2,2,2,2,2,2,  3,3,3,3,3]
}

# Mapping: {0: "CLICK", 1: "TYPE", 2: "NAVIGATE", 3: "VERIFY"}

# 2. Tokenization
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

dataset = Dataset.from_dict(raw_data)
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# 3. Training Arguments (For PFE Performance)
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,        # Zid f'el epochs bech DistilBERT ya7fedh mriguel
    per_device_train_batch_size=4,
    weight_decay=0.01,
    logging_dir='./logs',
)

# 4. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# 5. START TRAINING & SAVE
print("🚀 Training IntentClassifier...")
trainer.train()
# Create directory if it doesn't exist
save_path = "trained_models/nlp"
os.makedirs(save_path, exist_ok=True)

# Save the model
torch.save(model.state_dict(), os.path.join(save_path, "intention_classifier.pt"))
print(f"✅ intention_classifier.pt is ready at {save_path}!")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/25 [00:00<?, ? examples/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


🚀 Training IntentClassifier...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ intention_classifier.pt is ready at trained_models/nlp!
